# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joj48/Flyrank.ai-SEO-Project/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
import duckdb

# Attempt to load HF_TOKEN from Colab Secrets first, then environment variables
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    HF_TOKEN = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found! Please add 'HF_TOKEN' to Colab Secrets (🔑) "
        "and enable 'Notebook access', or set it as an environment variable."
    )

# Connect to DuckDB and configure Hugging Face token
con = duckdb.connect()
con.execute(f"""
    CREATE SECRET IF NOT EXISTS hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    );
""")

print("Successfully authenticated DuckDB with Hugging Face!")

Successfully authenticated DuckDB with Hugging Face!


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
PERFORMANCE_TABLE = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

In [5]:
# Verify Grain Uniqueness and Time Boundaries
query_grain = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date || '||' || client_hash_id || '||' || content_hash_id)) AS distinct_grain_keys,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{PERFORMANCE_TABLE}'
WHERE month = '2026-03';
"""

df_grain = con.execute(query_grain).df()
print(df_grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  distinct_grain_keys   min_date   max_date
0     9841378              9841378 2026-03-01 2026-03-31


### Unit of Analysis & Time Window Definition

* **Grain (One Row):** One row represents a unique `(report_date, client_hash_id, content_hash_id)` triplet, recording daily performance metrics (GSC clicks/impressions and GA4 session data) for a specific piece of content belonging to a client on a single calendar day.
* **Development Time Window:** Mid-panel month of **March 2026** (`month = '2026-03'`), spanning from `2026-03-01` to `2026-03-31` across **9,841,378 total rows**.
* **Sealed Window:** June 2026 (`month = '2026-06'`) is strictly reserved as the final sealed test set.
* **Verification Proof:** As shown in the query output below, `total_rows` (9,841,378) matches `distinct_grain_keys` (9,841,378) exactly, confirming the primary key has zero duplicate rows across the 31-day window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Categorization Strategy

We sort all fields from `fact_content_daily_performance` into four explicit buckets:

#### 1. Context (Identifiers & Grain Keys)
These define the unit of analysis and allow joins/aggregations without being fed into the model as predictors:
* `report_date`: The calendar day of observation.
* `client_hash_id`: Anonymized unique identifier for the client account.
* `content_hash_id`: Anonymized unique identifier for the piece of content/landing page.

#### 2. Features (Predictors knowable at decision moment $t$)
Constructed strictly using historical trailing windows ($t-7$ to $t-1$) prior to prediction date $t$:
* `f1_rolling_7d_clicks`: Trailing 7-day sum of GSC clicks (`gsc_clicks`).
* `f2_rolling_7d_impressions`: Trailing 7-day sum of GSC impressions (`gsc_impressions`).
* `f3_avg_position_3d`: Trailing 3-day average GSC rank position (`gsc_avg_position`).
* `f4_rolling_7d_ga4_sessions`: Trailing 7-day sum of organic/total sessions (`ga4_sessions`).
* `f5_past_ctr`: Trailing 7-day aggregate CTR (`f1_rolling_7d_clicks / f2_rolling_7d_impressions`).

#### 3. Label / Target (Outcome measured in future window $t$ to $t+7$)
* `target_label`: A binary indicator ($1/0$) set to $1$ if content receives $>10$ total GSC clicks over the next 7 days ($t$ to $t+7$).

#### 4. Excluded Fields (with explicit rationale)
* `gsc_clicks` (same-day raw metric): **Excluded** from features at decision moment $t$ because same-day clicks overlap with the target evaluation period ($t$ to $t+7$) and create direct data leakage.
* `gsc_sum_position` & `ga4_total_engagement_sec`: **Excluded** as redundant raw metrics in favor of their normalized counterparts (`gsc_avg_position` and aggregated session totals).
* AI referral columns (`ai_chatgpt`, `ai_perplexity`, `ai_gemini`, etc.): **Excluded** due to high sparsity (~95%+ zero values in early dataset partitions).

In [6]:
# Preview representative fields from all four buckets
query_fields_preview = f"""
SELECT
    -- Context
    report_date,
    client_hash_id,
    content_hash_id,

    -- Feature Inputs (Raw)
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,

    -- Quality Context
    gsc_data_available,

    -- Excluded Sparse Field Example
    ai_gemini
FROM '{PERFORMANCE_TABLE}'
WHERE month = '2026-03' AND gsc_data_available IS TRUE
LIMIT 5;
"""

df_fields_preview = con.execute(query_fields_preview).df()
df_fields_preview

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,gsc_data_available,ai_gemini
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,True,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,True,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,True,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,True,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,True,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification Statements & Queries

We prove every contract claim with explicit DuckDB queries on the `2026-03` mid-panel month:

1. **Grain Proof**: Verify that `(report_date, client_hash_id, content_hash_id)` has zero duplicate combinations.
2. **Date Span & Window Proof**: Confirm the dataset strictly bounds `2026-03-01` to `2026-03-31`.
3. **Availability / Quality Filter Proof**: Check how many rows survive using the explicit quality condition `gsc_data_available IS TRUE`.
4. **Missing Value Proof**: Prove zero missing values exist across key context identifiers (`report_date`, `client_hash_id`, `content_hash_id`).

In [7]:
# Query 1: Verify Grain Uniqueness, Row Count, and Boundary Window
query_grain_and_window = f"""
SELECT
    COUNT(*) AS total_raw_rows,
    COUNT(DISTINCT (report_date || '||' || client_hash_id || '||' || content_hash_id)) AS unique_grain_keys,
    COUNT(*) - COUNT(DISTINCT (report_date || '||' || client_hash_id || '||' || content_hash_id)) AS duplicate_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT report_date) AS total_active_days
FROM '{PERFORMANCE_TABLE}'
WHERE month = '2026-03';
"""

# Query 2: Availability Filter Check (IS TRUE) and Null Value Sanity Check
query_availability_and_nulls = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) AS surviving_rows_gsc_true,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS survival_pct,
    COUNT(CASE WHEN report_date IS NULL THEN 1 END) AS null_dates,
    COUNT(CASE WHEN client_hash_id IS NULL THEN 1 END) AS null_clients,
    COUNT(CASE WHEN content_hash_id IS NULL THEN 1 END) AS null_content
FROM '{PERFORMANCE_TABLE}'
WHERE month = '2026-03';
"""

print("--- 1. Grain & Window Verification ---")
df_grain_win = con.execute(query_grain_and_window).df()
display(df_grain_win)

print("\n--- 2. Availability (IS TRUE) & Null Value Verification ---")
df_avail_null = con.execute(query_availability_and_nulls).df()
display(df_avail_null)

--- 1. Grain & Window Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_raw_rows,unique_grain_keys,duplicate_count,min_date,max_date,total_active_days
0,9841378,9841378,0,2026-03-01,2026-03-31,31



--- 2. Availability (IS TRUE) & Null Value Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,surviving_rows_gsc_true,survival_pct,null_dates,null_clients,null_content
0,9841378,3611061,36.69,0,0,0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data Limitations & Boundary Constraints

Even with clean grain verification, this dataset slice has inherent structural limits that define what it can and *cannot* tell us:

1. **Window Overlap & Cold-Start Bias**:
   * Calculating trailing features (e.g., $t-7$ to $t-1$) requires 7 prior days of historical records.
   * Rows in the first week of the month (`2026-03-01` to `2026-03-07`) lack complete trailing history unless warmed up across partition boundaries. Cold-start content items appearing mid-panel will have skewed or zero-padded features.

2. **Integration Asymmetry (GSC-only vs GA4 Integration)**:
   * Google Search Console metrics (`gsc_data_available`) and Google Analytics metrics (`ga4_data_available`) do not have 100% mutual availability across all client accounts.
   * The dataset cannot tell us on-page user engagement for content pieces where Search Console is present but GA4 integration is disabled or unlinked (`client_has_ga4 IS FALSE`).

3. **Inability to Predict Long-Tail Anonymized Traffic**:
   * Google Search Console suppresses low-volume or privacy-sensitive queries.
   * Aggregated metrics at the page/content level cannot reveal individual query intent for ultra-long-tail search impressions, creating an unobserved tail in traffic attribution.

In [8]:
# 1. Measure Cold-Start / Early-Window Rows (First 7 Days of Month)
query_cold_start = f"""
SELECT
    COUNT(CASE WHEN report_date < '2026-03-08' THEN 1 END) AS early_window_rows,
    COUNT(CASE WHEN report_date >= '2026-03-08' THEN 1 END) AS fully_warmed_rows,
    ROUND(COUNT(CASE WHEN report_date < '2026-03-08' THEN 1 END) * 100.0 / COUNT(*), 2) AS early_window_pct
FROM '{PERFORMANCE_TABLE}'
WHERE month = '2026-03';
"""

# 2. Measure Integration Asymmetry (GSC vs GA4 Availability Discrepancy)
query_asymmetry = f"""
SELECT
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) AS both_available,
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS FALSE THEN 1 END) AS gsc_only,
    COUNT(CASE WHEN gsc_data_available IS FALSE AND ga4_data_available IS TRUE THEN 1 END) AS ga4_only,
    ROUND(COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS FALSE THEN 1 END) * 100.0 / COUNT(*), 2) AS gsc_only_pct
FROM '{PERFORMANCE_TABLE}'
WHERE month = '2026-03';
"""

print("--- 1. Cold-Start / Window Overlap Proof ---")
display(con.execute(query_cold_start).df())

print("\n--- 2. Integration Asymmetry Proof (GSC vs GA4) ---")
display(con.execute(query_asymmetry).df())

--- 1. Cold-Start / Window Overlap Proof ---


,early_window_rows,fully_warmed_rows,early_window_pct
0,2111744,7729634,21.46



--- 2. Integration Asymmetry Proof (GSC vs GA4) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,both_available,gsc_only,ga4_only,gsc_only_pct
0,364347,1718348,49619,17.46


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.